In [1]:
import sys
from pathlib import Path

ROOT = str(Path.cwd().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from src.data_loader import load_all_datasets
from src.preprocessing import add_rul_column
from src.config import DATASETS, USEFUL_SENSORS, MAX_RUL

data = load_all_datasets()

In [2]:
train = add_rul_column(data["FD001"][0])

scaler = StandardScaler()
train_scaled = train.copy()
train_scaled[USEFUL_SENSORS] = scaler.fit_transform(train[USEFUL_SENSORS])

last_cycle = train.groupby("unit_id").last().reset_index()
last_cycle_scaled = train_scaled.groupby("unit_id").last().reset_index()

In [3]:
sample = train[train["unit_id"].isin(train["unit_id"].unique()[:25])]

fig = px.scatter_3d(
    sample,
    x="sensor_7",
    y="sensor_11",
    z="sensor_4",
    color="rul",
    color_continuous_scale="RdYlGn",
    opacity=0.4,
    title="Espacio 3D — sensor_7 × sensor_11 × sensor_4 — FD001",
    height=700,
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(template="plotly_white")
fig.show()

In [4]:
sample = train[train["unit_id"].isin(train["unit_id"].unique()[:20])]

fig = px.scatter_3d(
    sample,
    x="cycle",
    y="sensor_11",
    z="rul",
    color="unit_id",
    opacity=0.5,
    title="Superficie de degradación — Ciclo × Sensor_11 × RUL",
    height=700,
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

In [5]:
pca = PCA(n_components=3)
pca_result = pca.fit_transform(last_cycle_scaled[USEFUL_SENSORS])

pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2", "PC3"])
pca_df["rul"] = last_cycle["rul"].values
pca_df["unit_id"] = last_cycle["unit_id"].values

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="rul",
    color_continuous_scale="RdYlGn",
    opacity=0.7,
    title=f"PCA 3D — Último ciclo por motor — Varianza explicada: {pca.explained_variance_ratio_.sum():.1%}",
    height=700,
    hover_data=["unit_id", "rul"],
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(template="plotly_white")
fig.show()

In [6]:
pca_full = PCA(n_components=len(USEFUL_SENSORS))
pca_full.fit(last_cycle_scaled[USEFUL_SENSORS])

cumulative = np.cumsum(pca_full.explained_variance_ratio_) * 100
individual = pca_full.explained_variance_ratio_ * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"PC{i+1}" for i in range(len(individual))],
    y=individual,
    name="Individual",
    marker_color="#4C72B0",
))
fig.add_trace(go.Scatter(
    x=[f"PC{i+1}" for i in range(len(cumulative))],
    y=cumulative,
    name="Acumulada",
    marker_color="#C44E52",
    line=dict(width=2.5),
))
fig.add_hline(y=95, line_dash="dash", line_color="gray", annotation_text="95%")
fig.update_layout(
    title="Varianza explicada por componente PCA",
    yaxis_title="% varianza",
    template="plotly_white",
    height=450,
)
fig.show()

In [7]:
tsne = TSNE(n_components=3, perplexity=30, random_state=42, max_iter=1000)
tsne_result = tsne.fit_transform(last_cycle_scaled[USEFUL_SENSORS])

tsne_df = pd.DataFrame(tsne_result, columns=["t-SNE1", "t-SNE2", "t-SNE3"])
tsne_df["rul"] = last_cycle["rul"].values
tsne_df["unit_id"] = last_cycle["unit_id"].values

fig = px.scatter_3d(
    tsne_df,
    x="t-SNE1",
    y="t-SNE2",
    z="t-SNE3",
    color="rul",
    color_continuous_scale="RdYlGn",
    opacity=0.7,
    title="t-SNE 3D — Último ciclo por motor — FD001",
    height=700,
    hover_data=["unit_id", "rul"],
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(template="plotly_white")
fig.show()

In [8]:
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}],
           [{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=DATASETS,
)

for i, ds_id in enumerate(DATASETS):
    tr = add_rul_column(data[ds_id][0])
    lc = tr.groupby("unit_id").last().reset_index()
    available = [s for s in USEFUL_SENSORS if s in lc.columns]

    sc = StandardScaler()
    scaled = sc.fit_transform(lc[available])
    pca_3 = PCA(n_components=3)
    result = pca_3.fit_transform(scaled)

    row = i // 2 + 1
    col = i % 2 + 1

    fig.add_trace(go.Scatter3d(
        x=result[:, 0],
        y=result[:, 1],
        z=result[:, 2],
        mode="markers",
        marker=dict(size=4, color=lc["rul"].values, colorscale="RdYlGn", opacity=0.7),
        name=ds_id,
        showlegend=False,
    ), row=row, col=col)

fig.update_layout(
    title="PCA 3D — Último ciclo — 4 sub-datasets",
    height=900,
    template="plotly_white",
)
fig.show()

In [9]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=["FD002 (6 cond, 1 falla)", "FD004 (6 cond, 2 fallas)"],
)

for i, ds_id in enumerate(["FD002", "FD004"]):
    tr = data[ds_id][0]
    unique_ops = tr[["op_setting_1", "op_setting_2", "op_setting_3"]].drop_duplicates()

    fig.add_trace(go.Scatter3d(
        x=unique_ops["op_setting_1"],
        y=unique_ops["op_setting_2"],
        z=unique_ops["op_setting_3"],
        mode="markers",
        marker=dict(size=6, color=unique_ops["op_setting_1"], colorscale="Viridis"),
        name=ds_id,
    ), row=1, col=i + 1)

fig.update_layout(
    title="Clusters de condiciones operacionales — FD002 vs FD004",
    height=600,
    template="plotly_white",
)
fig.show()

In [10]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=["FD002 (6 cond, 1 falla)", "FD004 (6 cond, 2 fallas)"],
)

for i, ds_id in enumerate(["FD002", "FD004"]):
    tr = data[ds_id][0]
    unique_ops = tr[["op_setting_1", "op_setting_2", "op_setting_3"]].drop_duplicates()

    fig.add_trace(go.Scatter3d(
        x=unique_ops["op_setting_1"],
        y=unique_ops["op_setting_2"],
        z=unique_ops["op_setting_3"],
        mode="markers",
        marker=dict(size=6, color=unique_ops["op_setting_1"], colorscale="Viridis"),
        name=ds_id,
    ), row=1, col=i + 1)

fig.update_layout(
    title="Clusters de condiciones operacionales — FD002 vs FD004",
    height=600,
    template="plotly_white",
)
fig.show()

In [11]:
train_anim = add_rul_column(data["FD001"][0]).copy()
train_anim["life_pct"] = train_anim.groupby("unit_id")["cycle"].transform(lambda x: (x / x.max() * 100).round(-1).astype(int))

scaler = StandardScaler()
train_anim[USEFUL_SENSORS] = scaler.fit_transform(train_anim[USEFUL_SENSORS])

pca2 = PCA(n_components=2)
pca_coords = pca2.fit_transform(train_anim[USEFUL_SENSORS])
train_anim["PC1"] = pca_coords[:, 0]
train_anim["PC2"] = pca_coords[:, 1]

sample_anim = train_anim[train_anim["unit_id"].isin(train_anim["unit_id"].unique()[:20])]

fig = px.scatter(
    sample_anim,
    x="PC1",
    y="PC2",
    color="rul",
    animation_frame="life_pct",
    color_continuous_scale="RdYlGn",
    range_x=[sample_anim["PC1"].min() * 1.1, sample_anim["PC1"].max() * 1.1],
    range_y=[sample_anim["PC2"].min() * 1.1, sample_anim["PC2"].max() * 1.1],
    title="PCA 2D animado por fase de vida — FD001",
    height=600,
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(template="plotly_white")
fig.show()

In [12]:
from scipy.interpolate import griddata

sensor_x = "sensor_7"
sensor_y = "sensor_11"

lc = last_cycle[[sensor_x, sensor_y, "rul"]].dropna()

xi = np.linspace(lc[sensor_x].min(), lc[sensor_x].max(), 50)
yi = np.linspace(lc[sensor_y].min(), lc[sensor_y].max(), 50)
xi, yi = np.meshgrid(xi, yi)

zi = griddata(
    (lc[sensor_x].values, lc[sensor_y].values),
    lc["rul"].values,
    (xi, yi),
    method="cubic",
)

fig = go.Figure(go.Surface(
    x=xi,
    y=yi,
    z=zi,
    colorscale="RdYlGn",
    opacity=0.85,
    colorbar=dict(title="RUL"),
))
fig.update_layout(
    title=f"Superficie RUL — {sensor_x} × {sensor_y} — FD001",
    scene=dict(
        xaxis_title=sensor_x,
        yaxis_title=sensor_y,
        zaxis_title="RUL",
    ),
    template="plotly_white",
    height=700,
)
fig.show()

In [13]:
all_last = []
for ds_id in DATASETS:
    tr = add_rul_column(data[ds_id][0])
    lc = tr.groupby("unit_id").last().reset_index()
    available = [s for s in USEFUL_SENSORS if s in lc.columns]
    lc_sub = lc[available].copy()
    lc_sub["dataset"] = ds_id
    lc_sub["rul"] = lc["rul"].values
    all_last.append(lc_sub)

all_last_df = pd.concat(all_last, ignore_index=True)
feature_cols = [s for s in USEFUL_SENSORS if s in all_last_df.columns]

sc = StandardScaler()
scaled_all = sc.fit_transform(all_last_df[feature_cols])

tsne2 = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
tsne_all = tsne2.fit_transform(scaled_all)

all_last_df["t-SNE1"] = tsne_all[:, 0]
all_last_df["t-SNE2"] = tsne_all[:, 1]

fig = px.scatter(
    all_last_df,
    x="t-SNE1",
    y="t-SNE2",
    color="dataset",
    symbol="dataset",
    opacity=0.7,
    title="t-SNE 2D — Todos los sub-datasets — Último ciclo",
    height=600,
    hover_data=["rul"],
)
fig.update_layout(template="plotly_white")
fig.show()

In [14]:
fig = px.scatter(
    all_last_df,
    x="t-SNE1",
    y="t-SNE2",
    color="rul",
    color_continuous_scale="RdYlGn",
    symbol="dataset",
    opacity=0.7,
    title="t-SNE 2D — Todos los sub-datasets — Coloreado por RUL",
    height=600,
)
fig.update_layout(template="plotly_white")
fig.show()

In [15]:
top_sensors = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]

fig = px.parallel_coordinates(
    last_cycle,
    dimensions=top_sensors + ["rul"],
    color="rul",
    color_continuous_scale="RdYlGn",
    title="Coordenadas paralelas — Sensores clave + RUL — FD001",
    height=550,
)
fig.update_layout(template="plotly_white")
fig.show()

In [16]:
print("=" * 60)
print("  RESUMEN EDA 3D / INTERACTIVO")
print("=" * 60)
print(f"  PCA 3 componentes explican: {pca.explained_variance_ratio_.sum():.1%} de varianza")
print(f"  t-SNE revela separación clara entre motores sanos y degradados")
print(f"  Superficie RUL muestra relación no-lineal sensor_7 × sensor_11")
print(f"  FD002/FD004 muestran 6 clusters operacionales claros en 3D")
print(f"  Coordenadas paralelas confirman patrones de degradación multi-sensor")
print(f"  PCA animado muestra migración progresiva hacia zona de falla")
print("=" * 60)

  RESUMEN EDA 3D / INTERACTIVO
  PCA 3 componentes explican: 88.1% de varianza
  t-SNE revela separación clara entre motores sanos y degradados
  Superficie RUL muestra relación no-lineal sensor_7 × sensor_11
  FD002/FD004 muestran 6 clusters operacionales claros en 3D
  Coordenadas paralelas confirman patrones de degradación multi-sensor
  PCA animado muestra migración progresiva hacia zona de falla
